# PROTOTYPE — Qwen3.8 FP8 R8-L2 root-first lifecycle qualification

**Question:** after the unchanged R8-B2 warmup and frozen request, does sending TERM only to the SGLang root let its coordinated shutdown close the complete externally observed process tree, port listener, GPU applications, and teardown log by the frozen deadline?

This notebook preserves the root-only TERM treatment and repairs only the external observer: exact timeout identity, post-TERM PID/PGID reuse guards, identity-bound group-KILL containment, completed-scan deadline adjudication, and a quiet window anchored to the first completed closed scan. It runs no ARC game, submits nothing, and tests no concurrency.

In [ ]:
import base64
from pathlib import Path

Path('/tmp/PROTOTYPE-qwen3_8_r8_l2_probe.py').write_bytes(
    base64.b64decode('...')
)
Path('/tmp/PROTOTYPE-qwen3_8_fp8-manifest.json').write_bytes(
    base64.b64decode('...')
)


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

input_root = Path('/kaggle/input')
wheel_paths = list(input_root.rglob('sglang-0.5.19.dev591+g8e04f66a7-cp312-cp312-linux_x86_64.whl'))
lock_paths = [
    path
    for path in input_root.rglob('requirements.lock')
    if sha256_file(path) == 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c'
]
if len(wheel_paths) != 1 or len(lock_paths) != 1:
    raise RuntimeError(
        'expected one pinned SGLang wheel and lockfile in attached inputs, '
        f'found wheels={wheel_paths}, locks={lock_paths}'
    )
wheel_path = wheel_paths[0]
lock_path = lock_paths[0]
if sha256_file(wheel_path) != '9d07f8bf3b718dc4cd970e72a03415b0b0597da89625acbee7a655d6387e7a8d':
    raise RuntimeError('mounted SGLang wheel does not match the pinned bytes')
wheelhouse = wheel_path.parent
runtime_provenance = Path('/tmp/sglang-runtime-provenance.json')
runtime_provenance.write_text(json.dumps({
    'dataset_source': 'mahdiseddigh/kaggle-t4-stack-2026-08-28',
    'dataset_version': 1,
    'repository': 'https://github.com/sgl-project/sglang.git',
    'commit': '8e04f66a7046fa083cda3f5b8e8325632a5af8fd',
    'artifact_path': str(wheel_path),
    'artifact_sha256': '9d07f8bf3b718dc4cd970e72a03415b0b0597da89625acbee7a655d6387e7a8d',
    'lock_path': str(lock_path),
    'lock_sha256': 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c',
}, indent=2) + '\n')

bootstrap_dir = Path('/kaggle/working/qwen38-fp8-r8l2-bootstrap')
bootstrap_dir.mkdir(parents=True, exist_ok=False)
overrides_path = Path('/tmp/sglang-runtime-overrides.txt')
overrides_path.write_text(
    '\n'.join([
        'flashinfer-python==0.6.17',
        'numba==0.65.1',
        'compressed-tensors==0.18.0',
        'outlines==1.3.3',
        'opencv-python-headless==5.0.0.93',
        'openai==3.5.0',
        '',
    ])
)
uv_install_command = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-index',
    '--find-links',
    str(wheelhouse),
    'uv==0.12.7',
]
with (bootstrap_dir / 'runtime-install.log').open('w') as install_log:
    subprocess.run(
        uv_install_command,
        check=True,
        stdout=install_log,
        stderr=subprocess.STDOUT,
    )
uv_executable = shutil.which('uv')
if not uv_executable:
    raise RuntimeError('uv executable missing after offline installation')
install_command = [
    uv_executable,
    'pip',
    'install',
    '--system',
    '--no-index',
    '--find-links',
    str(wheelhouse),
    '--override',
    str(overrides_path),
    '--upgrade',
    str(wheel_path),
    'torch==2.13.0+cu130',
    'transformers==5.12.1',
    'flashinfer-python==0.6.17',
    'sglang-kernel==0.4.6.post1',
    'triton==3.7.1',
    'cuda-tile==1.6.0rc5',
    'flash-attn-4==4.0.0b19',
    'opentelemetry-semantic-conventions==0.65b0',
]
(bootstrap_dir / 'runtime-install-command.json').write_text(
    json.dumps({
        'uv_install': uv_install_command,
        'runtime_install': install_command,
        'overrides': overrides_path.read_text().splitlines(),
    }, indent=2) + '\n'
)
with (bootstrap_dir / 'runtime-install.log').open('a') as install_log:
    subprocess.run(
        install_command,
        check=True,
        stdout=install_log,
        stderr=subprocess.STDOUT,
    )
(bootstrap_dir / 'runtime-provenance.json').write_text(
    runtime_provenance.read_text()
)
with (bootstrap_dir / 'runtime-packages.json').open('w') as package_log:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'list', '--format', 'json'],
        check=True,
        stdout=package_log,
    )


In [ ]:
import subprocess
import sys
from pathlib import Path

output_dir = Path('/kaggle/working/qwen38-fp8-r8l2-probe')
probe_command = [
        sys.executable,
        '/tmp/PROTOTYPE-qwen3_8_r8_l2_probe.py',
        '--output-dir',
        str(output_dir),
        '--manifest',
        '/tmp/PROTOTYPE-qwen3_8_fp8-manifest.json',
        '--input-root',
        '/kaggle/input',
        '--model-source',
        'jakobbrggen/qwen3-8-27b-fp8-hf-snapshot',
        '--runtime-provenance',
        str(runtime_provenance),
        '--startup-timeout-s',
        '1200',
        '--request-timeout-s',
        '180',
    ]
probe_command.extend(['--attention-backend', 'triton', '--sampling-backend', 'pytorch', '--profile-imports', '--r8-l2'])
subprocess.run(
    probe_command,
    check=True,
)
print((output_dir / 'result.json').read_text())
